In [6]:

# TASK 3: DATA PROCESSING & FEATURE ENGINEERING PIPELINE (NO LEAKAGE)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

# Load Raw Dataset
df = pd.read_csv('ardd_fatalities.csv', low_memory=False)

# Drop structural artifact
if 'Unnamed: 23' in df.columns:
    df = df.drop(columns=['Unnamed: 23'])

# Initial Data Cleaning & Target Definition
# Clean Age column and isolate drivers/riders >= 17
df['Age'] = df['Age'].replace([-9, '-9', '-9.0', 'Unknown'], np.nan)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df = df.dropna(subset=['Age'])
df = df[df['Age'] >= 17].copy()

# Target: Youth (17-25) = 1, Older (26+) = 0
df['Target_Youth'] = (df['Age'] <= 25).astype(int)

# Select Features & Handle Sentinel Values
feature_cols = [
    'State', 'Speed Limit', 'Crash Type', 'Bus Involvement', 
    'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement', 
    'Dayweek', 'Time of day'
]

X = df[feature_cols].copy()
y = df['Target_Youth'].values

# Replace sentinel values with NaN across all features
X = X.replace([-9, '-9', '-9.0', ' -9', 'Unknown', 'unknown'], np.nan)

# Feature Engineering: Create Risk Indicators
# Clean Time of day to extract hour if available or create Night flag
if 'Time of day' in X.columns:
    X['Time_Clean'] = pd.to_datetime(X['Time of day'], format='%H:%M:%S', errors='coerce').dt.hour
    X['Is_Night'] = X['Time_Clean'].apply(lambda x: 1 if (x >= 22 or x <= 5) else 0)
    X = X.drop(columns=['Time_Clean'])

# Leakage-Free Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit Imputers & Encoders ONLY on Training Data
# Categorical columns imputation & encoding
cat_cols = X_train.select_dtypes(include=['string','object', 'category']).columns.tolist()

imputer_cat = SimpleImputer(strategy='most_frequent')
X_train[cat_cols] = imputer_cat.fit_transform(X_train[cat_cols])
X_test[cat_cols] = imputer_cat.transform(X_test[cat_cols])

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

# Numeric columns imputation
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
imputer_num = SimpleImputer(strategy='median')
X_train[num_cols] = imputer_num.fit_transform(X_train[num_cols])
X_test[num_cols] = imputer_num.transform(X_test[num_cols])

# Convert back to contiguous numpy arrays
X_train_clean = X_train.values.astype(np.float32)
X_test_clean = X_test.values.astype(np.float32)

# Address Class Imbalance via SMOTE (Applied ONLY to Training Set)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_clean, y_train)

print("\n=== TASK 3 PREPROCESSING COMPLETED ===\n")
print(f"Original Training Shape:  {X_train_clean.shape}, Class Balance: {np.bincount(y_train)}")
print(f"Resampled Training Shape: {X_train_resampled.shape}, Class Balance: {np.bincount(y_train_resampled)}")
print(f"Test Shape (Unmodified):  {X_test_clean.shape}, Class Balance: {np.bincount(y_test)}")


=== TASK 3 PREPROCESSING COMPLETED ===

Original Training Shape:  (40837, 9), Class Balance: [29428 11409]
Resampled Training Shape: (58856, 9), Class Balance: [29428 29428]
Test Shape (Unmodified):  (10210, 9), Class Balance: [7358 2852]


In [4]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn"])

0